In [0]:
import json
from pyspark.sql.functions import col

# 1. Configuration Constants
COURSE_ID = "py_eng"
COURSE_TITLE = "Python English"
COURSE_DESCRIPTION = "Python Basics to Advance Full Course"
# COURSE_IMAGE = "https://upload.wikimedia.org/wikipedia/commons/c/c3/Python-logo-notext.svg"
SHARED_NOTES_ID = "py_notes_shared"
# Professional disclaimer
DISCLAIMER = "This course aggregates highly-rated YouTube tutorials to provide a structured learning path. All video content is hosted on YouTube, ensuring that creators receive full credit, engagement, and views for their valuable contributions."

# 2. Fetch Data from Silver Table
# Sort by topic_id to keep the chapters in order
silver_df = spark.sql(f"SELECT main_topic_id, main_topic_name, sub_videos_json FROM courseify.default.course_silver WHERE course_id = '{COURSE_ID}' ORDER BY main_topic_id")
silver_records = silver_df.collect()

# 3. Build the Chapters List
chapters_list = []

for row in silver_records:
    topic_id = row['main_topic_id']
    topic_title = row['main_topic_name']
    
    # Parse the sub_videos_json string back to a Python list of dictionaries
    sub_videos = []
    if row['sub_videos_json']:
        try:
            raw_videos = json.loads(row['sub_videos_json'])
            
            # Format each video object
            for vid in raw_videos:
                formatted_vid = {
                    "video_title": vid.get("refined_title", "Unknown Title"),
                    "video_link": f"{vid.get('video_id')}",
                    "channel": vid.get("channel", "Unknown Channel")
                }
                sub_videos.append(formatted_vid)
        except Exception as e:
            print(f"Error parsing JSON for topic {topic_id}: {e}")
            
    # Create the chapter object
    chapter = {
        "id": topic_id,
        "title": topic_title,
        "videos": sub_videos  # Array of videos for this topic
    }
    chapters_list.append(chapter)

# 4. Construct the Final JSON Object
final_course_json = {
    "id": COURSE_ID,
    "title": COURSE_TITLE,
    "description": COURSE_DESCRIPTION,
    "shared_notes_id": SHARED_NOTES_ID,
    "disclaimer": DISCLAIMER, 
    "chapters": chapters_list
}

# 5. Output the result
# json.dumps converts the Python dictionary into a formatted JSON string
output_json_string = json.dumps(final_course_json, indent=4)

print("✅ Final JSON Structure Generated Successfully!\n")
print(output_json_string)

file_name = "py_eng.json"
with open(file_name, "w") as json_file:
    json_file.write(output_json_string)

# Optional: Nuvvu deenni oka file laga save cheyalante
# dbutils.fs.put("dbfs:/FileStore/courseify/final_py_eng_course.json", output_json_string, overwrite=True)